# ENGR 1451/2451 — Weekly Assignment 10
**10 points**

This assignment focuses on three goals:
(i) interpreting Type I and Type II errors, sensitivity, and positive predictive value using **real biopsy-image data**;
(ii) investigating how significance cutoffs and sample size affect power, using measured features from the same dataset; and
(iii) comparing independent groups and paired observations with $t$-tests.


**Due:** See Canvas.

Complete the work directly in this notebook. Show your Python code and give brief written explanations where requested. Submit the completed notebook as directed in Canvas.

## What to hand in

Work this assignment with the **course tutor** in the class project. That is the AI help to use on this assignment, and it keeps the record for you — you do not write one yourself.

At the end of each working session, type **"make my record"** in the chat. Copy the block it produces and paste it into the **Session records** cell near the bottom of this notebook.

You do not have to finish in one sitting. To pick up later, start a new chat in the same project, say which assignment you are on and which problem you stopped at, and keep going. That session gives you its own block, which you paste under the previous one.

Records are not graded for correctness or for how much help you needed, and extra credit is given for a record that shows you attempted the problem. The code and the written answers still have to be yours.

## LE10-1. Real data: diagnostic flags and base rates (3 pts)

**Dataset:** `./data/LE10_wisconsin_tumors.csv` contains **569 real, labeled biopsy-image observations** from the Wisconsin Diagnostic Breast Cancer dataset. Each row represents a breast mass, with a known diagnosis (`benign` or `malignant`) and measurements extracted from images of cell nuclei. You will use the feature `mean_radius`.

For this classroom exercise, suppose a simple imaging rule **flags** a case when `mean_radius >= 14.0`. A flag means **the rule predicts malignant**; a non-flag means it predicts benign. The threshold is chosen for teaching, **not** as a validated medical screening rule.

The data come from a selected group of diagnostic cases, **not a random population-screening sample**. We will therefore distinguish (1) rates calculated from these real observations from (2) projections under a *hypothetical* population prevalence.

**Source:** [UCI Breast Cancer Wisconsin (Diagnostic)](https://archive.ics.uci.edu/dataset/17/breast-cancer-wisconsin-diagnostic), Wolberg, Mangasarian, Street & Street (1993), DOI: 10.24432/C5DW2B (CC BY 4.0). This teaching CSV contains original real observations but only two of the measured features.

### A. Build the screening table and examine prevalence (2 pts)

Using Python:

1. Load the CSV with `pd.read_csv()` and show the number of cases in each diagnosis category.
2. Using the fixed threshold above, count **true positives (TP)**, **false negatives (FN)**, **false positives (FP)**, and **true negatives (TN)**. Treat *malignant* as the positive class.
3. Display a $2\times 2$ `pd.DataFrame` with **Flagged / Not flagged** as rows and **Malignant / Benign** as columns.
4. Calculate the observed **sensitivity**, **specificity**, **positive predictive value (PPV)**, and **accuracy**:

$$
\mathrm{Sensitivity}=\frac{TP}{TP+FN},\quad
\mathrm{Specificity}=\frac{TN}{TN+FP},\quad
\mathrm{PPV}=\frac{TP}{TP+FP},\quad
\mathrm{Accuracy}=\frac{TP+TN}{TP+TN+FP+FN}.
$$

5. **Base-rate thought experiment:** Suppose this rule is applied to **200,000** future cases in a hypothetical population where **0.5%** are malignant. Assume the sensitivity and specificity found in this CSV would transfer unchanged. Compute the *expected* TP, FN, FP, and TN, then the projected PPV. Expected counts need not be whole numbers; round only when displaying them. Compare the projected PPV with the PPV from the observed dataset.

Report percentages to **one decimal place**.


In [48]:
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.precision", 4)

cases = pd.read_csv("./data/wisconsin_tumors.csv")
display(cases.head())
print(cases["diagnosis"].value_counts())

threshold = 14.0 # A case is flagged when mean_radius >= threshold.

# -------------- Your code here
# Calculate TP, FN, FP, TN, then display the 2 x 2 screening table.
malignant_cases = cases.loc[cases["diagnosis"] == "malignant"]
benign_cases = cases.loc[cases["diagnosis"] == "benign"]

true_positives = len(malignant_cases.loc[malignant_cases["mean_radius"] >= threshold]) #correctly flagged as malignant
false_positives = len(benign_cases.loc[benign_cases["mean_radius"] >= threshold]) #incorrectly flagged as malignant
true_negatives = len(benign_cases.loc[benign_cases["mean_radius"] < threshold]) #correctly flagged as benign
false_negatives = len(malignant_cases.loc[malignant_cases["mean_radius"] < threshold]) #incorrectly flagged as benign

screening_table = pd.DataFrame({
    "Flagged":[true_positives,false_positives],
    "Not Flagged":[false_negatives,true_negatives],
}, index=["Malignant","Benign"])

# Compute empirical sensitivity, specificity, PPV, and accuracy.
sensitivity_emp = true_positives/(true_positives + false_negatives)
specificity_emp = true_negatives/(true_negatives + false_positives)
ppv_emp = true_positives/(true_positives + false_positives)
accuracy_emp = (true_positives + true_negatives)/(true_positives + true_negatives + false_positives + false_negatives)

# Project expected screening outcomes for 200,000 cases with 0.5% malignancy.
total_cases_emp = len(cases)
cases_exp = 200000
percent_malignant = 0.5
proportion_malignant = percent_malignant/100
malignant_exp = proportion_malignant*cases_exp
benign_exp = cases_exp - malignant_exp

tp_exp = sensitivity_emp * malignant_exp #detection rate (sensitivity) * total malignant cases
fn_exp = malignant_exp - tp_exp #malignant cases that were not detected
tn_exp = specificity_emp * benign_exp #specificity * total benign cases
fp_exp = benign_exp - tn_exp #benign cases that were incorrectly detected


screening_table_exp = pd.DataFrame({
    "Flagged":[tp_exp,fp_exp],
    "Not Flagged":[fn_exp,tn_exp],
}, index=["Malignant","Benign"])

# Compute the projected PPV (using unrounded expected counts).
ppv_exp = tp_exp/(tp_exp + fp_exp)

screening_table.head()
display_table = pd.concat([screening_table.T,screening_table_exp.T],keys=["Empirical","Expected"])

print("\n\nEmpirical"
    "\nSenisitivity:",round(sensitivity_emp,1),
    "\nSpecificity:",round(specificity_emp,1),
    "\nPPV:",round(ppv_emp,1),
    "\nAccuracy:",round(accuracy_emp,1),
    "\n\nExpected",
    "\nPPV:",round(ppv_exp,1))
print("Unrounded PPV:",ppv_exp)

display_table.head()

,diagnosis,mean_radius,mean_smoothness
0,malignant,17.99,0.1184
1,malignant,20.57,0.0847
2,malignant,19.69,0.1096
3,malignant,11.42,0.1425
4,malignant,20.29,0.1003


diagnosis
benign       357
malignant    212
Name: count, dtype: int64


Empirical
Senisitivity: 0.8 
Specificity: 0.9 
PPV: 0.8 
Accuracy: 0.9 

Expected 
PPV: 0.0
Unrounded PPV: 0.028458359388424264


Malignant       Benign
Empirical Flagged       180.0000      52.0000
          Not Flagged    32.0000     305.0000
Expected  Flagged       849.0566   28985.9944
          Not Flagged   150.9434  170014.0056

### B. Interpret the results (1 pt)

Answer in about **3–4 sentences**:

1. For this screening rule, what is a **Type I error** (false positive)? What is a **Type II error** (false negative)?
2. Why does the PPV change so much when the assumed prevalence changes, even if sensitivity and specificity stay fixed? Does a specificity near 85% imply that 85% of flagged cases are malignant?
3. Why should you **not** claim that the observed proportion of malignant cases in this dataset is the prevalence in the general population?

**Answer:**
1. Type 1 error: a flagged tumor is benign, Type 2 error: a non-flagged tumor is malignant
2. The PPV value changes so much because of the fact that there is now a smaller percentage of malignant tumors (thus a higher proportion of false positives) than previously, leading the ppv (the chance of a true positive--correctly identifiying a tumor a malignant--given a flagged tumor) to decrease. A specificity does not imply that 85% of flagged cases are malignant, rather it represents that 85% of benign cases are not flagged.
3. Given that the sample is not random, it is not fair to claim that the observed proportion of malignant cases is the prevalance of the overall population. There is a possibility that people who get biopsies already have a higher suspicion (and potentially likleyhood) of having a malignant tumor.


## LE10-2. Significance cutoffs, $\beta$, and power (3 pts)

Use the **same real CSV**, but now study another imaging feature, `mean_smoothness`. We are designing a **population-level test based on the mean of $n$ independent biopsies**, *not* screening an individual case. Unlike LE10-1, which uses a fixed threshold to classify individual cases, we now choose cutoffs to control the probability of a Type I error at a specified significance level, $\alpha$.

Estimate the mean and sample SD of `mean_smoothness` separately for **benign** and **malignant** cases in the CSV. For this *planning exercise*, treat those estimated values as fixed benchmarks. Under a **normal approximation for sample means**, model

$$
\bar X\mid H_0 \approx N\!\left(\mu_0,\frac{\sigma_0^2}{n}\right),\qquad
\bar X\mid H_A \approx N\!\left(\mu_1,\frac{\sigma_1^2}{n}\right),
$$

where $H_0$ uses the **benign** reference distribution and the specified $H_A$ uses the **malignant** distribution. A false alarm means rejecting the benign reference when it is true; a miss means failing to reject it when the malignant model is true. **These are approximate design calculations**, not an evaluation of a clinical test or a claim that all biopsy measurements are normally distributed.

Use a **two-sided** decision rule: flag a sample if its mean is below the lower cutoff **or** above the upper cutoff. Compute cutoffs from the *benign* sampling distribution; compute $\beta$ as the probability of falling **between those cutoffs under the malignant model**.

| Plan | Independent biopsies $n$ | Significance level $\alpha$ |
|---|---:|---:|
| A | 10 | 0.05 |
| B | 10 | 0.01 |
| C | 25 | 0.01 |

### A. Calculate error rates and power (2 pts)

Using Python:

1. Extract `mean_smoothness` for benign and malignant cases; calculate each group's sample mean and sample SD using `.std(ddof=1)`.
2. For each plan, define `benign_model` and `malignant_model` with `stats.norm(loc=..., scale=SD/np.sqrt(n))`.
3. Calculate the **lower and upper cutoffs** using `benign_model.ppf([alpha/2, 1-alpha/2])`.
4. Compute $\beta= P(\text{not flagged}\mid H_A)$ using `malignant_model.cdf(upper) - malignant_model.cdf(lower)` and **power** $=1-\beta$.
5. Display a table with plan, $n$, $\alpha$, both cutoffs, $\beta$, and power. Report probabilities to **three decimal places**.

*Use the actual measurements to estimate model parameters; do not simulate additional biopsy records.*


In [ ]:
# Use the dataset loaded in LE10-1.
benign = cases.loc[cases["diagnosis"] == "benign", "mean_smoothness"]
malignant = cases.loc[cases["diagnosis"] == "malignant", "mean_smoothness"]

# Calculate mu0, sd0 (benign), and mu1, sd1 (malignant) from the CSV.
# Print both pairs of estimates.

plans = pd.DataFrame({
    "plan": ["A", "B", "C"],
    "n": [10, 10, 25],
    "alpha": [0.05, 0.01, 0.01],
})

# For each plan:
#   - Define benign_model and malignant_model using stats.norm(...).
#   - Calculate cutoffs using benign_model.ppf(...).
#   - Calculate beta using malignant_model.cdf(...).
#   - Calculate power = 1 - beta.
# Display a labeled results table.

# Your code here


### B. Explain the tradeoffs (1 pt)

Answer in about **2–3 sentences**:

1. Compare plans **A and B**: What happens to false alarms ($\alpha$) and missed detections ($\beta$) when the cutoff becomes stricter but $n$ stays the same?
2. Compare plans **B and C**: How does using more **independent biopsies** increase power at the same $\alpha$? Why would this be a study-design consideration rather than a way to improve diagnosis of a *single* patient?

**Answer:**


## LE10-3. Comparing two mean temperatures (4 pts)

Choose the analysis based on the **study design**, as in w10b. Both parts concern peak battery-module temperature during a standardized load test; **lower is better**.

*The datasets in this problem are simulated classroom examples, not measurements from a published study.*

### A. Independent modules: new vs. standard cooling (2.5 pts)

The file `./data/LE10_battery_cooling.csv` contains peak temperatures ($^\circ$C) for **26 different battery modules**: 12 were randomly assigned to **new** cooling and 14 to **standard** cooling. Each module was measured once.

Let $\Delta=\mu_{\mathrm{new}}-\mu_{\mathrm{standard}}$. We want to know whether the new method **reduces** the population mean peak temperature. Test the boundary null $H_0:\Delta=0$ against $H_A:\Delta<0$, using $\alpha=0.05$.

Assume observations from different modules are independent, group distributions are approximately normal without strong outliers, and population SDs are unknown. Use **Welch's** two-sample $t$-test (do not assume equal population variances).

In [ ]:
cooling = pd.read_csv("./data/battery_cooling.csv")

# The order is new minus standard: a negative difference favors the new method.
new = cooling.loc[cooling["method"] == "new", "peak_temperature_C"].to_numpy()
standard = cooling.loc[cooling["method"] == "standard", "peak_temperature_C"].to_numpy()

display(cooling.head())
print(cooling["method"].value_counts())

,module_id,method,peak_temperature_C
0,N01,new,60.1
1,N02,new,63.0
2,N03,new,59.4
3,N04,new,62.2
4,N05,new,57.8


method
standard    14
new         12
Name: count, dtype: int64


Use Python to:

1. Calculate each group's **sample size, mean, and sample SD** (with `ddof=1`), and the observed difference $\bar x_{\mathrm{new}}-\bar x_{\mathrm{standard}}$.
2. Calculate the estimated SE of the difference:

$$
\widehat{SE}=\sqrt{\frac{s_{\mathrm{new}}^2}{n_{\mathrm{new}}}+\frac{s_{\mathrm{standard}}^2}{n_{\mathrm{standard}}}}.
$$

3. Run `stats.ttest_ind(new, standard, equal_var=False, alternative="less")`. Print the **$t$ statistic, one-sided $P$-value**, and decision at $\alpha=0.05$.
4. For context, run the same Welch test with `alternative="two-sided"` and use its `.confidence_interval(confidence_level=0.95)` method to report a **two-sided 95% confidence interval** for $\Delta$. (You do not need to calculate Welch degrees of freedom manually.)

**Answer:**

In [ ]:
# Calculate group summaries (n, sample mean, sample SD).
# Calculate the observed difference and estimated SE.
# Run Welch's one-sided test and print the decision at alpha = 0.05.
# Run a two-sided Welch test and print its 95% confidence interval.

# Your code here

### B. Matched modules: before vs. after a firmware change (1.5 pts)

A **separate experiment** measures the *same eight modules* before and after a cooling-control firmware update. The arrays are aligned: `before[i]` and `after[i]` belong to the same module.

1. Calculate the eight within-module differences `after - before` and their mean.
2. Run `stats.ttest_rel(after, before, alternative="less")` to test $H_0:\mu_d=0$ against $H_A:\mu_d<0$, where $d=\text{after}-\text{before}$. Print $t$, the one-sided $P$-value, and the decision at $\alpha=0.05$.
3. In **2–3 sentences total**, interpret the conclusions in parts A and B. Explain **why a paired rather than independent test** is required in part B and whether a statistically significant result by itself establishes the engineering importance of a temperature change.

**Answer:**

In [ ]:
# Same modules measured twice; each index identifies the same module.
before = np.array([72, 68, 71, 70, 74, 69, 73, 67])
after  = np.array([69, 65, 70, 67, 73, 66, 72, 66])

# Calculate and display within-module differences and their mean.
# Run the paired t-test, report t and P, and make the decision at alpha = 0.05.

# Your code here

## Session records

At the end of each working session, type **"make my record"** in the chat and paste the block it produces into the cell below. If you work across several sessions, paste each new block underneath the previous one.

Session records are not graded for correctness or for how much help you needed, and extra credit is given for a record that shows you attempted the problem. They are read to find out which steps are hardest, and used to improve the course.

<!-- Paste your session record blocks in this cell, one after another. -->

*(session records go here)*
{
  "schema_version": "engr-tutor-export-2",
  "assignment": "WA10",
  "export_n": 1,
  "session_date": "2026-11-08",
  "messages": [
    {"i": 1, "who": "student", "code": true, "text": "Can you help me troubleshoot my code? \n\n```python\nimport numpy as np\nimport pandas as pd\nfrom scipy import stats\n\npd.set_option(\"display.precision\", 4)\n\ncases = pd.read_csv(\"./data/wisconsin_tumors.csv\")\ndisplay(cases.head())\nprint(cases[\"diagnosis\"].value_counts())\n\nthreshold = 14.0 # A case is flagged when mean_radius >= threshold.\n\n# -------------- Your code here\n# Calculate TP, FN, FP, TN, then display the 2 x 2 screening table.\nmalignant_cases = cases.loc[cases[\"diagnosis\"] == \"malignant\"]\nbenign_cases = cases.loc[cases[\"diagnosis\"] == \"benign\"]\n\ntrue_positives = malignant_cases.loc(malignant_cases[\"mean_radius\"] >= threshold).count() #correctly flagged as malignant\nfalse_positives = malignant_cases.loc(malignant_cases[\"mean_radius\"] < threshold).count() #incorrectly flagged as malignant\ntrue_negatives = benign_cases.loc(benign_cases[\"mean_radius\"] < threshold).count() #correctly flagged as benign\nfalse_negatives = benign_cases.loc(benign_cases[\"mean_radius\"] >= threshold).count() #incorrectly flagged as benign\n\nscreening_table = pd.DataFrame({\n    \"True\":[true_positives,true_negatives],\n    \"False\":[false_positives,false_negatives],\n}, index=[\"Positive\",\"Negative\"])\n\n# Compute empirical sensitivity, specificity, PPV, and accuracy.\nsensitivity_emp = true_positives/(true_positives + false_negatives)\nspecificity_emp = true_negatives/(true_negatives + false_positives)\nppv_emp = true_positives/(true_positives + false_positives)\naccuracy_emp = (true_positives + true_negatives)/(true_positives + true_negatives + false_positives + false_negatives)\n\n# Project expected screening outcomes for 200,000 cases with 0.5% malignancy.\ntotal_cases_emp = len(cases)\ncases_exp = 200000\npercent_malignant = 0.5\nproportion_malignant = percent_malignant/100\nmalignant_exp = proportion_malignant*cases_exp\nbenign_exp = (1-proportion_malignant)*cases_exp\n\ntp_exp = (true_positives/total_cases_emp)*malignant_exp\nfp_exp = (false_positives/total_cases_emp)*malignant_exp\ntn_exp = (true_negatives/total_cases_emp)*benign_cases\nfn_exp = (false_negatives/total_cases_emp)*benign_cases\n\n# Compute the projected PPV (using unrounded expected counts).\nppv_exp = tp_exp/(tn_exp + fp_exp)\n\nscreening_table\n\n```\n\n---------------------------------------------------------------------------\nTypeError                                 Traceback (most recent call last)\nCell In[5], line 18\n     14 # Calculate TP, FN, FP, TN, then display the 2 x 2 screening table.\n     15 malignant_cases = cases.loc[cases[\"diagnosis\"] == \"malignant\"]\n     16 benign_cases = cases.loc[cases[\"diagnosis\"] == \"benign\"]\n     17 \n---> 18 true_positives = malignant_cases.loc(malignant_cases[\"mean_radius\"] >= threshold).count() #correctly flagged as malignant\n     19 false_positives = malignant_cases.loc(malignant_cases[\"mean_radius\"] < threshold).count() #incorrectly flagged as malignant\n     20 true_negatives = benign_cases.loc(benign_cases[\"mean_radius\"] < threshold).count() #correctly flagged as benign\n     21 false_negatives = benign_cases.loc(benign_cases[\"mean_radius\"] >= threshold).count() #incorrectly flagged as benign\n\nFile ~\\AppData\\Local\\Python\\pythoncore-3.14-64\\Lib\\site-packages\\pandas\\core\\indexing.py:766, in _LocationIndexer.__call__(self, axis)\n    763 new_self = type(self)(self.name, self.obj)\n    765 if axis is not None:\n--> [766](https://file+.vscode-resource.vscode-cdn.net/c%3A/Users/julia/Documents/GitHub/Julian-Brady-ENGR-1451-Fork/1-assignments/LE10/~/AppData/Local/Python/pythoncore-3.14-64/Lib/site-packages/pandas/core/indexing.py:766)     axis_int_none = self.obj._get_axis_number(axis)\n    767 else:\n    768     axis_int_none = axis\n\nFile ~\\AppData\\Local\\Python\\pythoncore-3.14-64\\Lib\\site-packages\\pandas\\core\\generic.py:524, in NDFrame._get_axis_number(cls, axis)\n    520     @classmethod\n    521     def _get_axis_number(cls, axis: Axis) -> AxisInt:\n    522         try:\n--> 523             return cls._AXIS_TO_AXIS_NUMBER[axis]\n--> [524](https://file+.vscode-resource.vscode-cdn.net/c%3A/Users/julia/Documents/GitHub/Julian-Brady-ENGR-1451-Fork/1-assignments/LE10/~/AppData/Local/Python/pythoncore-3.14-64/Lib/site-packages/pandas/core/generic.py:524)         except KeyError as err:\n    525             raise ValueError(\n    526                 f\"No axis named {axis} for object type {cls.__name__}\"\n    527             ) from err\n\nTypeError: cannot use 'pandas.Series' as a dict key (unhashable type: 'Series')"},
    {"i": 2, "who": "tutor", "text": "[trimmed]"},
    {"i": 3, "who": "student", "text": "Is there a way to display two dataframes side by side"},
    {"i": 4, "who": "tutor", "text": "[trimmed]"},
    {"i": 5, "who": "student", "text": "Is there a way to concatenate two screening tables?"},
    {"i": 6, "who": "tutor", "text": "[trimmed]"},
    {"i": 7, "who": "student", "text": "Is there a way to have categories of rows in a dataframe?"},
    {"i": 8, "who": "tutor", "text": "[trimmed]"},
    {"i": 9, "who": "student", "text": "Does this carry over to display"},
    {"i": 10, "who": "tutor", "text": "[trimmed]"},
    {"i": 11, "who": "student", "text": "Can you visualize the inputs for the multiindexing command?"},
    {"i": 12, "who": "tutor", "text": "[trimmed] (a diagram of the pd.MultiIndex.from_tuples inputs was shown, with a short explanation)"},
    {"i": 13, "who": "student", "text": "Is there a way I can multiindex from existing dataframes?"},
    {"i": 14, "who": "tutor", "text": "[trimmed]"},
    {"i": 15, "who": "student", "text": "How would this look for my 2x2 display table if I wanted to have one for expected counts s well as empirical?"},
    {"i": 16, "who": "tutor", "text": "[trimmed]"},
    {"i": 17, "who": "student", "text": "Additionally, if I wanted to print many variables at once, which command can I use to set the display precision ahead of time and avoid using round() over and over again?"},
    {"i": 18, "who": "tutor", "text": "[trimmed]"},
    {"i": 19, "who": "student", "text": "How could I do the same for standard print commands?"},
    {"i": 20, "who": "tutor", "text": "[trimmed]"},
    {"i": 21, "who": "student", "text": "Understood"},
    {"i": 22, "who": "tutor", "text": "[trimmed]"},
    {"i": 23, "who": "student", "code": true, "text": "Did I miss anything?\n\n```python\nimport numpy as np\nimport pandas as pd\nfrom scipy import stats\n\npd.set_option(\"display.precision\", 4)\n\ncases = pd.read_csv(\"./data/wisconsin_tumors.csv\")\ndisplay(cases.head())\nprint(cases[\"diagnosis\"].value_counts())\n\nthreshold = 14.0 # A case is flagged when mean_radius >= threshold.\n\n# -------------- Your code here\n# Calculate TP, FN, FP, TN, then display the 2 x 2 screening table.\nmalignant_cases = cases.loc[cases[\"diagnosis\"] == \"malignant\"]\nbenign_cases = cases.loc[cases[\"diagnosis\"] == \"benign\"]\n\ntrue_positives = len(malignant_cases.loc[malignant_cases[\"mean_radius\"] >= threshold]) #correctly flagged as malignant\nfalse_positives = len(malignant_cases.loc[malignant_cases[\"mean_radius\"] < threshold]) #incorrectly flagged as malignant\ntrue_negatives = len(benign_cases.loc[benign_cases[\"mean_radius\"] < threshold]) #correctly flagged as benign\nfalse_negatives = len(benign_cases.loc[benign_cases[\"mean_radius\"] >= threshold]) #incorrectly flagged as benign\n\nscreening_table = pd.DataFrame({\n    \"True\":[true_positives,true_negatives],\n    \"False\":[false_positives,false_negatives],\n}, index=[\"Positive\",\"Negative\"])\n\n# Compute empirical sensitivity, specificity, PPV, and accuracy.\nsensitivity_emp = true_positives/(true_positives + false_negatives)\nspecificity_emp = true_negatives/(true_negatives + false_positives)\nppv_emp = true_positives/(true_positives + false_positives)\naccuracy_emp = (true_positives + true_negatives)/(true_positives + true_negatives + false_positives + false_negatives)\n\n# Project expected screening outcomes for 200,000 cases with 0.5% malignancy.\ntotal_cases_emp = len(cases)\ncases_exp = 200000\npercent_malignant = 0.5\nproportion_malignant = percent_malignant/100\nmalignant_exp = proportion_malignant*cases_exp\nbenign_exp = (1-proportion_malignant)*cases_exp\n\ntp_exp = (true_positives/total_cases_emp)*malignant_exp\nfp_exp = (false_positives/total_cases_emp)*malignant_exp\ntn_exp = (true_negatives/total_cases_emp)*benign_exp\nfn_exp = (false_negatives/total_cases_emp)*benign_exp\n\nscreening_table_exp = pd.DataFrame({\n    \"True\":[tp_exp,tn_exp],\n    \"False\":[fp_exp,fn_exp],\n}, index=[\"Positive\",\"Negative\"])\n\n# Compute the projected PPV (using unrounded expected counts).\nppv_exp = tp_exp/(tp_exp + fp_exp)\n\nscreening_table.head()\ndisplay_table = pd.concat([screening_table,screening_table_exp],keys=[\"Empirical\",\"Expected\"])\n\nprint(\"\\n\\nEmpirical\"\n    \"\\nSenisitivity:\",round(sensitivity_emp,1),\n    \"\\nSpecificity:\",round(specificity_emp,1),\n    \"\\nPPV:\",round(ppv_emp,1),\n    \"\\nAccuracy:\",round(accuracy_emp,1),\n    \"\\n\\nExpected\",\n    \"\\nPPV:\",round(ppv_exp,1))\n\ndisplay_table.head()\n```"},
    {"i": 24, "who": "tutor", "text": "[trimmed]"},
    {"i": 25, "who": "student", "code": true, "text": "```python\nimport numpy as np\nimport pandas as pd\nfrom scipy import stats\n\npd.set_option(\"display.precision\", 4)\n\ncases = pd.read_csv(\"./data/wisconsin_tumors.csv\")\ndisplay(cases.head())\nprint(cases[\"diagnosis\"].value_counts())\n\nthreshold = 14.0 # A case is flagged when mean_radius >= threshold.\n\n# -------------- Your code here\n# Calculate TP, FN, FP, TN, then display the 2 x 2 screening table.\nmalignant_cases = cases.loc[cases[\"diagnosis\"] == \"malignant\"]\nbenign_cases = cases.loc[cases[\"diagnosis\"] == \"benign\"]\n\ntrue_positives = len(malignant_cases.loc[malignant_cases[\"mean_radius\"] >= threshold]) #correctly flagged as malignant\nfalse_positives = len(malignant_cases.loc[malignant_cases[\"mean_radius\"] < threshold]) #incorrectly flagged as malignant\ntrue_negatives = len(benign_cases.loc[benign_cases[\"mean_radius\"] < threshold]) #correctly flagged as benign\nfalse_negatives = len(benign_cases.loc[benign_cases[\"mean_radius\"] >= threshold]) #incorrectly flagged as benign\n\nscreening_table = pd.DataFrame({\n    \"Flagged\":[true_positives,true_negatives],\n    \"Not Flagged\":[false_positives,false_negatives],\n}, index=[\"Malignant\",\"Benign\"])\n\n# Compute empirical sensitivity, specificity, PPV, and accuracy.\nsensitivity_emp = true_positives/(true_positives + false_negatives)\nspecificity_emp = true_negatives/(true_negatives + false_positives)\nppv_emp = true_positives/(true_positives + false_positives)\naccuracy_emp = (true_positives + true_negatives)/(true_positives + true_negatives + false_positives + false_negatives)\n\n# Project expected screening outcomes for 200,000 cases with 0.5% malignancy.\ntotal_cases_emp = len(cases)\ncases_exp = 200000\npercent_malignant = 0.5\nproportion_malignant = percent_malignant/100\nmalignant_exp = proportion_malignant*cases_exp\nbenign_exp = (1-proportion_malignant)*cases_exp\n\ntp_exp = (true_positives/total_cases_emp)*malignant_exp\nfp_exp = (false_positives/total_cases_emp)*malignant_exp\ntn_exp = (true_negatives/total_cases_emp)*benign_exp\nfn_exp = (false_negatives/total_cases_emp)*benign_exp\n\nscreening_table_exp = pd.DataFrame({\n    \"Flagged\":[tp_exp,tn_exp],\n    \"Not Flagged\":[fp_exp,fn_exp],\n}, index=[\"Malignant\",\"Benign\"])\n\n# Compute the projected PPV (using unrounded expected counts).\nppv_exp = tp_exp/(tp_exp + fp_exp)\n\nscreening_table.head()\ndisplay_table = pd.concat([screening_table,screening_table_exp],keys=[\"Empirical\",\"Expected\"])\n\nprint(\"\\n\\nEmpirical\"\n    \"\\nSenisitivity:\",round(sensitivity_emp,1),\n    \"\\nSpecificity:\",round(specificity_emp,1),\n    \"\\nPPV:\",round(ppv_emp,1),\n    \"\\nAccuracy:\",round(accuracy_emp,1),\n    \"\\n\\nExpected\",\n    \"\\nPPV:\",round(ppv_exp,1))\n\ndisplay_table.head()\n```\n\nCan you elaborate more on the TN/FN mixup? Wouldnt a true negative be a benign case that is lower than the threshold?"},
    {"i": 26, "who": "tutor", "text": "[trimmed]"},
    {"i": 27, "who": "student", "code": true, "text": "Is this better now? \n\n```python\ntrue_positives = len(malignant_cases.loc[malignant_cases[\"mean_radius\"] >= threshold]) #correctly flagged as malignant\nfalse_positives = len(malignant_cases.loc[benign_cases[\"mean_radius\"] >= threshold]) #incorrectly flagged as malignant\ntrue_negatives = len(benign_cases.loc[benign_cases[\"mean_radius\"] < threshold]) #correctly flagged as benign\nfalse_negatives = len(benign_cases.loc[malignant_cases[\"mean_radius\"] < threshold]) #incorrectly flagged as benign\n```"},
    {"i": 28, "who": "tutor", "text": "[trimmed]"},
    {"i": 29, "who": "student", "code": true, "text": "```python\nimport numpy as np\nimport pandas as pd\nfrom scipy import stats\n\npd.set_option(\"display.precision\", 4)\n\ncases = pd.read_csv(\"./data/wisconsin_tumors.csv\")\ndisplay(cases.head())\nprint(cases[\"diagnosis\"].value_counts())\n\nthreshold = 14.0 # A case is flagged when mean_radius >= threshold.\n\n# -------------- Your code here\n# Calculate TP, FN, FP, TN, then display the 2 x 2 screening table.\nmalignant_cases = cases.loc[cases[\"diagnosis\"] == \"malignant\"]\nbenign_cases = cases.loc[cases[\"diagnosis\"] == \"benign\"]\n\ntrue_positives = len(malignant_cases.loc[malignant_cases[\"mean_radius\"] >= threshold]) #correctly flagged as malignant\nfalse_positives = len(benign_cases.loc[benign_cases[\"mean_radius\"] >= threshold]) #incorrectly flagged as malignant\ntrue_negatives = len(benign_cases.loc[benign_cases[\"mean_radius\"] < threshold]) #correctly flagged as benign\nfalse_negatives = len(malignant_cases.loc[malignant_cases[\"mean_radius\"] < threshold]) #incorrectly flagged as benign\n\nscreening_table = pd.DataFrame({\n    \"Flagged\":[true_positives,true_negatives],\n    \"Not Flagged\":[false_positives,false_negatives],\n}, index=[\"Malignant\",\"Benign\"])\n\n# Compute empirical sensitivity, specificity, PPV, and accuracy.\nsensitivity_emp = true_positives/(true_positives + false_negatives)\nspecificity_emp = true_negatives/(true_negatives + false_positives)\nppv_emp = true_positives/(true_positives + false_positives)\naccuracy_emp = (true_positives + true_negatives)/(true_positives + true_negatives + false_positives + false_negatives)\n\n# Project expected screening outcomes for 200,000 cases with 0.5% malignancy.\ntotal_cases_emp = len(cases)\ncases_exp = 200000\npercent_malignant = 0.5\nproportion_malignant = percent_malignant/100\nmalignant_exp = proportion_malignant*cases_exp\nbenign_exp = (1-proportion_malignant)*cases_exp\n\ntp_exp = (true_positives/total_cases_emp)*malignant_exp\nfp_exp = (false_positives/total_cases_emp)*malignant_exp\ntn_exp = (true_negatives/total_cases_emp)*benign_exp\nfn_exp = (false_negatives/total_cases_emp)*benign_exp\n\nscreening_table_exp = pd.DataFrame({\n    \"Flagged\":[tp_exp,tn_exp],\n    \"Not Flagged\":[fp_exp,fn_exp],\n}, index=[\"Malignant\",\"Benign\"])\n\n# Compute the projected PPV (using unrounded expected counts).\nppv_exp = tp_exp/(tp_exp + fp_exp)\n\nscreening_table.head()\ndisplay_table = pd.concat([screening_table,screening_table_exp],keys=[\"Empirical\",\"Expected\"])\n\nprint(\"\\n\\nEmpirical\"\n    \"\\nSenisitivity:\",round(sensitivity_emp,1),\n    \"\\nSpecificity:\",round(specificity_emp,1),\n    \"\\nPPV:\",round(ppv_emp,1),\n    \"\\nAccuracy:\",round(accuracy_emp,1),\n    \"\\n\\nExpected\",\n    \"\\nPPV:\",round(ppv_exp,1))\n\ndisplay_table.head()\n```"},
    {"i": 30, "who": "tutor", "text": "[trimmed]"},
    {"i": 31, "who": "student", "code": true, "text": "```python\nimport numpy as np\nimport pandas as pd\nfrom scipy import stats\n\npd.set_option(\"display.precision\", 4)\n\ncases = pd.read_csv(\"./data/wisconsin_tumors.csv\")\ndisplay(cases.head())\nprint(cases[\"diagnosis\"].value_counts())\n\nthreshold = 14.0 # A case is flagged when mean_radius >= threshold.\n\n# -------------- Your code here\n# Calculate TP, FN, FP, TN, then display the 2 x 2 screening table.\nmalignant_cases = cases.loc[cases[\"diagnosis\"] == \"malignant\"]\nbenign_cases = cases.loc[cases[\"diagnosis\"] == \"benign\"]\n\ntrue_positives = len(malignant_cases.loc[malignant_cases[\"mean_radius\"] >= threshold]) #correctly flagged as malignant\nfalse_positives = len(benign_cases.loc[benign_cases[\"mean_radius\"] >= threshold]) #incorrectly flagged as malignant\ntrue_negatives = len(benign_cases.loc[benign_cases[\"mean_radius\"] < threshold]) #correctly flagged as benign\nfalse_negatives = len(malignant_cases.loc[malignant_cases[\"mean_radius\"] < threshold]) #incorrectly flagged as benign\n\nscreening_table = pd.DataFrame({\n    \"Flagged\":[true_positives,false_positives],\n    \"Not Flagged\":[true_negatives,false_negatives],\n}, index=[\"Malignant\",\"Benign\"])\n\n# Compute empirical sensitivity, specificity, PPV, and accuracy.\nsensitivity_emp = true_positives/(true_positives + false_negatives)\nspecificity_emp = true_negatives/(true_negatives + false_positives)\nppv_emp = true_positives/(true_positives + false_positives)\naccuracy_emp = (true_positives + true_negatives)/(true_positives + true_negatives + false_positives + false_negatives)\n\n# Project expected screening outcomes for 200,000 cases with 0.5% malignancy.\ntotal_cases_emp = len(cases)\ncases_exp = 200000\npercent_malignant = 0.5\nproportion_malignant = percent_malignant/100\nmalignant_exp = proportion_malignant*cases_exp\nbenign_exp = (1-proportion_malignant)*cases_exp\n\ntp_exp = (true_positives/total_cases_emp)*malignant_exp\nfp_exp = (false_positives/total_cases_emp)*malignant_exp\ntn_exp = (true_negatives/total_cases_emp)*benign_exp\nfn_exp = (false_negatives/total_cases_emp)*benign_exp\n\nscreening_table_exp = pd.DataFrame({\n    \"Flagged\":[tp_exp,fp_exp],\n    \"Not Flagged\":[tn_exp,fn_exp],\n}, index=[\"Malignant\",\"Benign\"])\n\n# Compute the projected PPV (using unrounded expected counts).\nppv_exp = tp_exp/(tp_exp + fp_exp)\n\nscreening_table.head()\ndisplay_table = pd.concat([screening_table,screening_table_exp],keys=[\"Empirical\",\"Expected\"])\n\nprint(\"\\n\\nEmpirical\"\n    \"\\nSenisitivity:\",round(sensitivity_emp,1),\n    \"\\nSpecificity:\",round(specificity_emp,1),\n    \"\\nPPV:\",round(ppv_emp,1),\n    \"\\nAccuracy:\",round(accuracy_emp,1),\n    \"\\n\\nExpected\",\n    \"\\nPPV:\",round(ppv_exp,1))\n\ndisplay_table.head()\n```"},
    {"i": 32, "who": "tutor", "text": "[trimmed]"},
    {"i": 33, "who": "student", "code": true, "text": "```python\nimport numpy as np\nimport pandas as pd\nfrom scipy import stats\n\npd.set_option(\"display.precision\", 4)\n\ncases = pd.read_csv(\"./data/wisconsin_tumors.csv\")\ndisplay(cases.head())\nprint(cases[\"diagnosis\"].value_counts())\n\nthreshold = 14.0 # A case is flagged when mean_radius >= threshold.\n\n# -------------- Your code here\n# Calculate TP, FN, FP, TN, then display the 2 x 2 screening table.\nmalignant_cases = cases.loc[cases[\"diagnosis\"] == \"malignant\"]\nbenign_cases = cases.loc[cases[\"diagnosis\"] == \"benign\"]\n\ntrue_positives = len(malignant_cases.loc[malignant_cases[\"mean_radius\"] >= threshold]) #correctly flagged as malignant\nfalse_positives = len(benign_cases.loc[benign_cases[\"mean_radius\"] >= threshold]) #incorrectly flagged as malignant\ntrue_negatives = len(benign_cases.loc[benign_cases[\"mean_radius\"] < threshold]) #correctly flagged as benign\nfalse_negatives = len(malignant_cases.loc[malignant_cases[\"mean_radius\"] < threshold]) #incorrectly flagged as benign\n\nscreening_table = pd.DataFrame({\n    \"Flagged\":[true_positives,false_positives],\n    \"Not Flagged\":[false_negatives,true_negatives],\n}, index=[\"Malignant\",\"Benign\"])\n\n# Compute empirical sensitivity, specificity, PPV, and accuracy.\nsensitivity_emp = true_positives/(true_positives + false_negatives)\nspecificity_emp = true_negatives/(true_negatives + false_positives)\nppv_emp = true_positives/(true_positives + false_positives)\naccuracy_emp = (true_positives + true_negatives)/(true_positives + true_negatives + false_positives + false_negatives)\n\n# Project expected screening outcomes for 200,000 cases with 0.5% malignancy.\ntotal_cases_emp = len(cases)\ncases_exp = 200000\npercent_malignant = 0.5\nproportion_malignant = percent_malignant/100\nmalignant_exp = proportion_malignant*cases_exp\nbenign_exp = (1-proportion_malignant)*cases_exp\n\ntp_exp = (true_positives/total_cases_emp)*malignant_exp\nfp_exp = (false_positives/total_cases_emp)*malignant_exp\ntn_exp = (true_negatives/total_cases_emp)*benign_exp\nfn_exp = (false_negatives/total_cases_emp)*benign_exp\n\nscreening_table_exp = pd.DataFrame({\n    \"Flagged\":[tp_exp,fp_exp],\n    \"Not Flagged\":[fn_exp,tn_exp],\n}, index=[\"Malignant\",\"Benign\"])\n\n# Compute the projected PPV (using unrounded expected counts).\nppv_exp = tp_exp/(tp_exp + fp_exp)\n\nscreening_table.head()\ndisplay_table = pd.concat([screening_table,screening_table_exp],keys=[\"Empirical\",\"Expected\"])\n\nprint(\"\\n\\nEmpirical\"\n    \"\\nSenisitivity:\",round(sensitivity_emp,1),\n    \"\\nSpecificity:\",round(specificity_emp,1),\n    \"\\nPPV:\",round(ppv_emp,1),\n    \"\\nAccuracy:\",round(accuracy_emp,1),\n    \"\\n\\nExpected\",\n    \"\\nPPV:\",round(ppv_exp,1))\n\ndisplay_table.head()\n```"},
    {"i": 34, "who": "tutor", "text": "[trimmed]"},
    {"i": 35, "who": "student", "code": true, "text": "```python\nimport numpy as np\nimport pandas as pd\nfrom scipy import stats\n\npd.set_option(\"display.precision\", 4)\n\ncases = pd.read_csv(\"./data/wisconsin_tumors.csv\")\ndisplay(cases.head())\nprint(cases[\"diagnosis\"].value_counts())\n\nthreshold = 14.0 # A case is flagged when mean_radius >= threshold.\n\n# -------------- Your code here\n# Calculate TP, FN, FP, TN, then display the 2 x 2 screening table.\nmalignant_cases = cases.loc[cases[\"diagnosis\"] == \"malignant\"]\nbenign_cases = cases.loc[cases[\"diagnosis\"] == \"benign\"]\n\ntrue_positives = len(malignant_cases.loc[malignant_cases[\"mean_radius\"] >= threshold]) #correctly flagged as malignant\nfalse_positives = len(benign_cases.loc[benign_cases[\"mean_radius\"] >= threshold]) #incorrectly flagged as malignant\ntrue_negatives = len(benign_cases.loc[benign_cases[\"mean_radius\"] < threshold]) #correctly flagged as benign\nfalse_negatives = len(malignant_cases.loc[malignant_cases[\"mean_radius\"] < threshold]) #incorrectly flagged as benign\n\nscreening_table = pd.DataFrame({\n    \"Flagged\":[true_positives,false_positives],\n    \"Not Flagged\":[false_negatives,true_negatives],\n}, index=[\"Malignant\",\"Benign\"])\n\n# Compute empirical sensitivity, specificity, PPV, and accuracy.\nsensitivity_emp = true_positives/(true_positives + false_negatives)\nspecificity_emp = true_negatives/(true_negatives + false_positives)\nppv_emp = true_positives/(true_positives + false_positives)\naccuracy_emp = (true_positives + true_negatives)/(true_positives + true_negatives + false_positives + false_negatives)\n\n# Project expected screening outcomes for 200,000 cases with 0.5% malignancy.\ntotal_cases_emp = len(cases)\ncases_exp = 200000\npercent_malignant = 0.5\nproportion_malignant = percent_malignant/100\nmalignant_exp = proportion_malignant*cases_exp\nbenign_exp = (1-proportion_malignant)*cases_exp\n\ntp_exp = (true_positives/total_cases_emp)*malignant_exp\nfp_exp = (false_positives/total_cases_emp)*malignant_exp\ntn_exp = (true_negatives/total_cases_emp)*benign_exp\nfn_exp = (false_negatives/total_cases_emp)*benign_exp\n\nscreening_table_exp = pd.DataFrame({\n    \"Flagged\":[tp_exp,fp_exp],\n    \"Not Flagged\":[fn_exp,tn_exp],\n}, index=[\"Malignant\",\"Benign\"])\n\n# Compute the projected PPV (using unrounded expected counts).\nppv_exp = tp_exp/(tp_exp + fp_exp)\n\nscreening_table.head()\ndisplay_table = pd.concat([screening_table.T,screening_table_exp.T],keys=[\"Empirical\",\"Expected\"])\n\nprint(\"\\n\\nEmpirical\"\n    \"\\nSenisitivity:\",round(sensitivity_emp,1),\n    \"\\nSpecificity:\",round(specificity_emp,1),\n    \"\\nPPV:\",round(ppv_emp,1),\n    \"\\nAccuracy:\",round(accuracy_emp,1),\n    \"\\n\\nExpected\",\n    \"\\nPPV:\",round(ppv_exp,1))\n\ndisplay_table.head()\n```"},
    {"i": 36, "who": "tutor", "text": "[trimmed]"},
    {"i": 37, "who": "student", "code": true, "text": "```python\ntp_exp = sensitivity_emp * malignant_exp #detection rate (sensitivity) * total malignant cases\nfn_exp = benign_exp - tn_exp #malignant cases that were not detected\ntn_exp = specificity_emp * benign_exp #specificity * total benign cases\nfp_exp = benign_exp - tn_exp #benign cases that were incorrectly detected\n\n```"},
    {"i": 38, "who": "tutor", "text": "[trimmed]"},
    {"i": 39, "who": "student", "code": true, "text": "```python\nimport numpy as np\nimport pandas as pd\nfrom scipy import stats\n\npd.set_option(\"display.precision\", 4)\n\ncases = pd.read_csv(\"./data/wisconsin_tumors.csv\")\ndisplay(cases.head())\nprint(cases[\"diagnosis\"].value_counts())\n\nthreshold = 14.0 # A case is flagged when mean_radius >= threshold.\n\n# -------------- Your code here\n# Calculate TP, FN, FP, TN, then display the 2 x 2 screening table.\nmalignant_cases = cases.loc[cases[\"diagnosis\"] == \"malignant\"]\nbenign_cases = cases.loc[cases[\"diagnosis\"] == \"benign\"]\n\ntrue_positives = len(malignant_cases.loc[malignant_cases[\"mean_radius\"] >= threshold]) #correctly flagged as malignant\nfalse_positives = len(benign_cases.loc[benign_cases[\"mean_radius\"] >= threshold]) #incorrectly flagged as malignant\ntrue_negatives = len(benign_cases.loc[benign_cases[\"mean_radius\"] < threshold]) #correctly flagged as benign\nfalse_negatives = len(malignant_cases.loc[malignant_cases[\"mean_radius\"] < threshold]) #incorrectly flagged as benign\n\nscreening_table = pd.DataFrame({\n    \"Flagged\":[true_positives,false_positives],\n    \"Not Flagged\":[false_negatives,true_negatives],\n}, index=[\"Malignant\",\"Benign\"])\n\n# Compute empirical sensitivity, specificity, PPV, and accuracy.\nsensitivity_emp = true_positives/(true_positives + false_negatives)\nspecificity_emp = true_negatives/(true_negatives + false_positives)\nppv_emp = true_positives/(true_positives + false_positives)\naccuracy_emp = (true_positives + true_negatives)/(true_positives + true_negatives + false_positives + false_negatives)\n\n# Project expected screening outcomes for 200,000 cases with 0.5% malignancy.\ntotal_cases_emp = len(cases)\ncases_exp = 200000\npercent_malignant = 0.5\nproportion_malignant = percent_malignant/100\nmalignant_exp = proportion_malignant*cases_exp\nbenign_exp = total_cases_emp - malignant_exp\n\ntp_exp = sensitivity_emp * malignant_exp #detection rate (sensitivity) * total malignant cases\nfn_exp = malignant_exp - tp_exp #malignant cases that were not detected\ntn_exp = specificity_emp * benign_exp #specificity * total benign cases\nfp_exp = benign_exp - tn_exp #benign cases that were incorrectly detected\n\n\nscreening_table_exp = pd.DataFrame({\n    \"Flagged\":[tp_exp,fp_exp],\n    \"Not Flagged\":[fn_exp,tn_exp],\n}, index=[\"Malignant\",\"Benign\"])\n\n# Compute the projected PPV (using unrounded expected counts).\nppv_exp = tp_exp/(tp_exp + fp_exp)\n\nscreening_table.head()\ndisplay_table = pd.concat([screening_table.T,screening_table_exp.T],keys=[\"Empirical\",\"Expected\"])\n\nprint(\"\\n\\nEmpirical\"\n    \"\\nSenisitivity:\",round(sensitivity_emp,1),\n    \"\\nSpecificity:\",round(specificity_emp,1),\n    \"\\nPPV:\",round(ppv_emp,1),\n    \"\\nAccuracy:\",round(accuracy_emp,1),\n    \"\\n\\nExpected\",\n    \"\\nPPV:\",round(ppv_exp,1))\n\ndisplay_table.head()\n```"},
    {"i": 40, "who": "tutor", "text": "[trimmed]"},
    {"i": 41, "who": "student", "code": true, "text": "```python\nimport numpy as np\nimport pandas as pd\nfrom scipy import stats\n\npd.set_option(\"display.precision\", 4)\n\ncases = pd.read_csv(\"./data/wisconsin_tumors.csv\")\ndisplay(cases.head())\nprint(cases[\"diagnosis\"].value_counts())\n\nthreshold = 14.0 # A case is flagged when mean_radius >= threshold.\n\n# -------------- Your code here\n# Calculate TP, FN, FP, TN, then display the 2 x 2 screening table.\nmalignant_cases = cases.loc[cases[\"diagnosis\"] == \"malignant\"]\nbenign_cases = cases.loc[cases[\"diagnosis\"] == \"benign\"]\n\ntrue_positives = len(malignant_cases.loc[malignant_cases[\"mean_radius\"] >= threshold]) #correctly flagged as malignant\nfalse_positives = len(benign_cases.loc[benign_cases[\"mean_radius\"] >= threshold]) #incorrectly flagged as malignant\ntrue_negatives = len(benign_cases.loc[benign_cases[\"mean_radius\"] < threshold]) #correctly flagged as benign\nfalse_negatives = len(malignant_cases.loc[malignant_cases[\"mean_radius\"] < threshold]) #incorrectly flagged as benign\n\nscreening_table = pd.DataFrame({\n    \"Flagged\":[true_positives,false_positives],\n    \"Not Flagged\":[false_negatives,true_negatives],\n}, index=[\"Malignant\",\"Benign\"])\n\n# Compute empirical sensitivity, specificity, PPV, and accuracy.\nsensitivity_emp = true_positives/(true_positives + false_negatives)\nspecificity_emp = true_negatives/(true_negatives + false_positives)\nppv_emp = true_positives/(true_positives + false_positives)\naccuracy_emp = (true_positives + true_negatives)/(true_positives + true_negatives + false_positives + false_negatives)\n\n# Project expected screening outcomes for 200,000 cases with 0.5% malignancy.\ntotal_cases_emp = len(cases)\ncases_exp = 200000\npercent_malignant = 0.5\nproportion_malignant = percent_malignant/100\nmalignant_exp = proportion_malignant*cases_exp\nbenign_exp = cases_exp - malignant_exp\n\ntp_exp = sensitivity_emp * malignant_exp #detection rate (sensitivity) * total malignant cases\nfn_exp = malignant_exp - tp_exp #malignant cases that were not detected\ntn_exp = specificity_emp * benign_exp #specificity * total benign cases\nfp_exp = benign_exp - tn_exp #benign cases that were incorrectly detected\n\n\nscreening_table_exp = pd.DataFrame({\n    \"Flagged\":[tp_exp,fp_exp],\n    \"Not Flagged\":[fn_exp,tn_exp],\n}, index=[\"Malignant\",\"Benign\"])\n\n# Compute the projected PPV (using unrounded expected counts).\nppv_exp = tp_exp/(tp_exp + fp_exp)\n\nscreening_table.head()\ndisplay_table = pd.concat([screening_table.T,screening_table_exp.T],keys=[\"Empirical\",\"Expected\"])\n\nprint(\"\\n\\nEmpirical\"\n    \"\\nSenisitivity:\",round(sensitivity_emp,1),\n    \"\\nSpecificity:\",round(specificity_emp,1),\n    \"\\nPPV:\",round(ppv_emp,1),\n    \"\\nAccuracy:\",round(accuracy_emp,1),\n    \"\\n\\nExpected\",\n    \"\\nPPV:\",round(ppv_exp,1))\n\ndisplay_table.head()\n```"},
    {"i": 42, "who": "tutor", "text": "[trimmed]"},
    {"i": 43, "who": "student", "text": "```markdown\n**Answer:**\n1. Type 1 error: benign tumor is flagged as malignant, Type 2 error: malignant tumor is flagged as benign\n2. The PPV value changes so much because of the fact that there is a higher proportion of false positives than previously, leading the ppv (the chance of a true positive, correctly identifiying a tumor a malignant) to decrease. A specificity does not imply that 85% of flagged cases are malignant.\n3. Different sampling methods may lead to an unrespresentative sample of the population of interest. For example, the rates of malignant tumors may be greater in the country, state, city, etc. where the sample was taken than the true prevalance of malignant tumors.\n\n```"},
    {"i": 44, "who": "tutor", "text": "[trimmed]"},
    {"i": 45, "who": "student", "text": "```markdown\n**Answer:**\n1. Type 1 error: a flagged tumor is benign, Type 2 error: a non-flagged tumor is malignant\n2. The PPV value changes so much because of the fact that there is now a smaller percentage of malignant tumors (thus a higher proportion of false positives) than previously, leading the ppv (the chance of a true positive, correctly identifiying a tumor a malignant) to decrease. A specificity does not imply that 85% of flagged cases are malignant.\n3. Given that the sample is not random, it is not fair to claim that the observed proportion of malignant cases is the prevalance of the overall population. There is a possibility that people who get biopsies already have a higher suspicion (and potentially likleyhood) of having a malignant tumor.\n\n```"},
    {"i": 46, "who": "tutor", "text": "[trimmed]"},
    {"i": 47, "who": "student", "text": "```markdown\n2. The PPV value changes so much because of the fact that there is now a smaller percentage of malignant tumors (thus a higher proportion of false positives) than previously, leading the ppv (the chance of a true positive--correctly identifiying a tumor a malignant--given a flagged tumor) to decrease. A specificity does not imply that 85% of flagged cases are malignant.\n```"},
    {"i": 48, "who": "tutor", "text": "[trimmed]"},
    {"i": 49, "who": "student", "text": "```markdown\nA specificity does not imply that 85% of flagged cases are malignant, rather it represents that 85% of benign cases are not flagged.\n```"},
    {"i": 50, "who": "tutor", "text": "[trimmed]"}
  ],
  "events": [
    {"i": 1, "part": "10-1A", "type": "code"},
    {"i": 3, "part": "10-1A", "type": "explain"},
    {"i": 5, "part": "10-1A", "type": "explain"},
    {"i": 7, "part": "10-1A", "type": "explain"},
    {"i": 9, "part": "10-1A", "type": "explain"},
    {"i": 11, "part": "10-1A", "type": "explain"},
    {"i": 13, "part": "10-1A", "type": "explain"},
    {"i": 15, "part": "10-1A", "type": "explain"},
    {"i": 17, "part": "10-1A", "type": "explain"},
    {"i": 19, "part": "10-1A", "type": "explain"},
    {"i": 23, "part": "10-1A", "type": "code"},
    {"i": 25, "part": "10-1A", "type": "code"},
    {"i": 27, "part": "10-1A", "type": "code"},
    {"i": 29, "part": "10-1A", "type": "code"},
    {"i": 31, "part": "10-1A", "type": "code"},
    {"i": 33, "part": "10-1A", "type": "code"},
    {"i": 35, "part": "10-1A", "type": "code"},
    {"i": 37, "part": "10-1A", "type": "code"},
    {"i": 39, "part": "10-1A", "type": "code"},
    {"i": 41, "part": "10-1A", "type": "code"},
    {"i": 50, "part": "10-1B", "type": "part_pass"}
  ],
  "parts": [
    {"part": "10-1A", "exchanges": 21, "max_tier": 3, "passed": false},
    {"part": "10-1B", "exchanges": 4, "max_tier": 2, "passed": true}
  ],
  "tutor_assessment": [
    "The student stalled longest in LE10-1A on the correct group for each of the four counts: false positives and false negatives took several rounds because the group a case came from and the mask used to select it did not match. After that came the placement of counts in the 2x2 table and the projection lines, where the expected counts divided by the whole dataset instead of by the group sizes and one benign_exp edit used the dataset size instead of the 200,000-case population. Presentation items (percent formatting, side-by-side PPV print, label capitalization, data file path) were still open at the end of the session.",
    "The student did not use the release valve or the unstuck option in this session. Many of the student's questions were free concept and Python questions about displaying and combining tables, MultiIndex, and print formatting, which were answered directly and did not count as attempts."
  ]
}

## Before you submit

1. Keep `LE10.ipynb` next to a `data` folder containing **both** `LE10_wisconsin_tumors.csv` and the **original** `LE10_battery_cooling.csv`. This revision provides the real Wisconsin CSV; the battery-cooling CSV from the earlier version of LE10 is still needed for Question 3.
2. **Restart Kernel and Run All Cells** after both CSV files are present and confirm every completed cell runs without errors.
3. Confirm your calculated tables, printed values, and written answers are visible in the saved notebook.
4. Confirm your session record blocks are pasted into the cell above.
5. Save the notebook, then upload the `.ipynb` to Canvas, as directed.
